# 12. 내구성 실행

## 학습 목표

- 내구성 실행(Durable Execution)의 개념과 필요성을 이해한다
- 체크포인터와 내구성 실행의 관계를 안다
- `@entrypoint` + `@task`로 내구성을 보장하는 방법을 익힌다
- 내구성 모드(exit, async, sync)의 차이를 이해한다
- 장애 시나리오에서의 복구 과정을 안다

## 12.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — https://lf.ddok.ai


## 12.2 내구성 실행 개념

**내구성 실행(Durable Execution)**이란 프로세스나 워크플로가 핵심 지점에서 진행 상태를 저장하여,  
일시 중지 후 나중에 정확히 중단된 위치에서 재개할 수 있는 기법입니다.

**왜 필요한가?**

| 시나리오 | 설명 |
|----------|------|
| 장애 복구 | 서버 장애 시 처음부터 다시 실행하지 않고 중단 지점에서 재개 |
| 상태 영속 | 긴 실행 시간을 가진 워크플로의 중간 결과를 보존 |
| Human-in-the-loop | 사람의 승인을 기다리는 동안 상태를 유지 |

LangGraph는 체크포인터(checkpointer)를 통해 내구성 실행을 지원합니다.

## 12.3 핵심 요구사항

내구성 실행을 구현하려면 세 가지 요소가 필요합니다:

1. **영속 계층 (Persistence Layer)**  
   체크포인터를 통해 워크플로 진행 상태를 기록합니다.  
   예: `InMemorySaver`(개발용), `PostgresSaver`(프로덕션용)

2. **스레드 식별자 (Thread ID)**  
   워크플로 인스턴스의 실행 기록을 추적하는 고유 ID입니다.  
   같은 `thread_id`를 사용하면 이전 실행을 이어서 재개할 수 있습니다.

3. **태스크 래핑 (Task Wrapping)**  
   비결정적(non-deterministic) 연산과 부수 효과(side-effect) 연산을  
   태스크로 감싸서 재개 시 재실행을 방지합니다.

## 12.4 내구성 모드 비교

LangGraph는 성능과 일관성 사이의 균형을 맞추는 세 가지 모드를 제공합니다:

| 모드 | 동작 | 트레이드오프 |
|------|------|------------|
| `"exit"` | 완료/에러/인터럽트 시에만 영속화 | 최고 성능, 중간 복구 불가 |
| `"async"` | 다음 단계 실행 중 비동기로 영속화 | 적절한 균형, 약간의 크래시 위험 |
| `"sync"` | 다음 단계 실행 전 동기로 영속화 | 최대 내구성, 성능 비용 |

대부분의 사용 사례에서는 기본 모드(`"exit"`)로 충분합니다.  
미션 크리티컬한 워크플로에서는 `"sync"` 모드를 고려하세요.

## 12.5 문제가 있는 코드

부수 효과(API 호출 등)를 태스크로 감싸지 않으면,  
재개 시 동일한 API 호출이 다시 실행될 수 있습니다.

In [3]:
# 문제가 있는 접근 방식: 부수 효과를 직접 호출
print("""# BAD: 부수 효과가 태스크로 감싸지지 않음
def call_api(state: State):
    # 이 API 호출은 재개 시 다시 실행됨!
    result = requests.get(state['url']).text[:100]
    return {"result": result}
""")
print("문제점:")
print("  1. 장애 후 재개 시 API가 다시 호출됨")
print("  2. 비결정적 결과가 달라질 수 있음")
print("  3. 중복 요청으로 부작용 발생 가능")

# BAD: 부수 효과가 태스크로 감싸지지 않음
def call_api(state: State):
    # 이 API 호출은 재개 시 다시 실행됨!
    result = requests.get(state['url']).text[:100]
    return {"result": result}

문제점:
  1. 장애 후 재개 시 API가 다시 호출됨
  2. 비결정적 결과가 달라질 수 있음
  3. 중복 요청으로 부작용 발생 가능


## 12.6 @task로 개선

`@task` 데코레이터로 부수 효과를 감싸면,  
재개 시 이전 결과를 체크포인트에서 복원하여 재실행을 방지합니다.

In [4]:
# 개선된 접근 방식: @task로 부수 효과 래핑
print("""# GOOD: @task로 부수 효과를 감쌈
from langgraph.func import task

@task
def _make_request(url: str):
    return requests.get(url).text[:100]

def call_api(state: State):
    # 각 요청이 개별 태스크로 실행됨
    requests = [_make_request(url) for url in state['urls']]
    results = [req.result() for req in requests]
    return {"results": results}
""")
print("개선 효과:")
print("  1. 재개 시 체크포인트에서 결과 복원")
print("  2. API 중복 호출 방지")
print("  3. 각 태스크가 독립적으로 추적됨")

# GOOD: @task로 부수 효과를 감쌈
from langgraph.func import task

@task
def _make_request(url: str):
    return requests.get(url).text[:100]

def call_api(state: State):
    # 각 요청이 개별 태스크로 실행됨
    requests = [_make_request(url) for url in state['urls']]
    results = [req.result() for req in requests]
    return {"results": results}

개선 효과:
  1. 재개 시 체크포인트에서 결과 복원
  2. API 중복 호출 방지
  3. 각 태스크가 독립적으로 추적됨


## 12.7 Graph API에서의 내구성

StateGraph에 체크포인터를 연결하면 각 노드 실행 후 상태가 자동 저장됩니다.

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class DocState(TypedDict):
    topic: str
    draft: str
    final: str


def write_draft(state: DocState) -> dict:
    return {"draft": f"Draft about {state['topic']}"}


def finalize(state: DocState) -> dict:
    return {"final": f"Final: {state['draft']}"}


checkpointer = InMemorySaver()

builder = StateGraph(DocState)
builder.add_node("write_draft", write_draft)
builder.add_node("finalize", finalize)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(checkpointer=checkpointer)

# 실행 (thread_id로 실행 추적)
config = {"configurable": {"thread_id": "doc-1"}}
result = graph.invoke({"topic": "LangGraph"}, config)
print("결과:", result)

결과: {'topic': 'LangGraph', 'draft': 'Draft about LangGraph', 'final': 'Final: Draft about LangGraph'}


## 12.8 Functional API에서의 내구성

`@entrypoint`와 `@task`를 조합하면 Functional API에서도 내구성을 보장할 수 있습니다.

In [6]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver


@task
def generate_draft(topic: str) -> str:
    return f"Draft about {topic}"


@task
def review_draft(draft: str) -> str:
    return f"Reviewed: {draft}"


func_checkpointer = InMemorySaver()


@entrypoint(checkpointer=func_checkpointer)
def write_document(topic: str) -> str:
    draft = generate_draft(topic).result()
    reviewed = review_draft(draft).result()
    return reviewed


config = {"configurable": {"thread_id": "func-1"}}
result = write_document.invoke("Durable Execution", config)
print("결과:", result)

결과: Reviewed: Draft about Durable Execution


## 12.9 장애 복구 시나리오

같은 `thread_id`로 재실행하면 체크포인트에서 이전 상태를 복원하여 이어서 실행합니다.

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class PipelineState(TypedDict):
    data: str
    step: int
    result: str


call_count = 0


def step_one(state: PipelineState) -> dict:
    global call_count
    call_count += 1
    print(f"  step_one 실행 (호출 횟수: {call_count})")
    return {"data": state["data"].upper(), "step": 1}


def step_two(state: PipelineState) -> dict:
    print(f"  step_two 실행")
    return {"result": f"Processed: {state['data']}", "step": 2}


recovery_saver = InMemorySaver()

builder = StateGraph(PipelineState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

pipeline = builder.compile(checkpointer=recovery_saver)

# 첫 번째 실행
config = {"configurable": {"thread_id": "recovery-1"}}
print("=== 첫 번째 실행 ===")
result = pipeline.invoke(
    {"data": "hello", "step": 0, "result": ""},
    config
)
print(f"결과: {result}")

# 체크포인트 확인
print("\n=== 체크포인트에서 상태 복원 ===")
saved = pipeline.get_state(config)
print(f"저장된 상태: {saved.values}")
print(f"step_one 총 호출 횟수: {call_count}")

=== 첫 번째 실행 ===
  step_one 실행 (호출 횟수: 1)
  step_two 실행
결과: {'data': 'HELLO', 'step': 2, 'result': 'Processed: HELLO'}

=== 체크포인트에서 상태 복원 ===
저장된 상태: {'data': 'HELLO', 'step': 2, 'result': 'Processed: HELLO'}
step_one 총 호출 횟수: 1


## 12.10 재개 시작점

워크플로가 재개될 때, API에 따라 시작점이 다릅니다:

| API | 재개 시작점 | 설명 |
|-----|-----------|------|
| StateGraph | 중단된 노드의 시작 | 해당 노드를 처음부터 다시 실행 |
| 서브그래프 | 부모 노드 → 서브그래프 내 중단 노드 | 부모 노드에서 시작 후 서브그래프 내 해당 노드로 이동 |
| Functional API | `@entrypoint` 시작 | 엔트리포인트에서 시작, `@task` 결과는 캐시에서 복원 |

**핵심 차이:**  
- StateGraph: 노드 단위 재개 (중단된 노드만 재실행)
- Functional API: 엔트리포인트부터 재실행하되, 완료된 `@task`는 캐시 결과 사용

## 12.11 프로덕션 내구성 패턴

프로덕션 환경에서 내구성을 보장하기 위한 모범 사례:

1. **멱등성(Idempotent) 연산 구현**  
   같은 요청을 여러 번 실행해도 결과가 동일하도록 설계합니다.  
   멱등성 키(idempotency key)를 활용하여 중복 처리를 방지합니다.

2. **부수 효과 분리**  
   API 호출, 파일 쓰기 등의 부수 효과를 개별 `@task`로 분리합니다.  
   순수 로직과 부수 효과를 명확히 구분합니다.

3. **비결정적 코드 래핑**  
   난수 생성, 타임스탬프 등 비결정적 연산도 `@task`로 감쌉니다.

4. **영속 저장소 사용**  
   개발: `InMemorySaver`  
   프로덕션: `PostgresSaver` 또는 외부 데이터베이스

5. **스레드 ID 관리**  
   각 워크플로 인스턴스에 고유한 `thread_id`를 부여합니다.  
   장애 복구 시 동일한 `thread_id`로 재개합니다.

## 12.12 요약

| 주제 | 핵심 내용 |
|------|----------|
| 내구성 개념 | 중단 지점에서 재개할 수 있는 실행 기법 |
| 핵심 요구사항 | 영속 계층 + 스레드 ID + 태스크 래핑 |
| 내구성 모드 | exit(기본), async(균형), sync(최대 내구성) |
| @task | 부수 효과를 감싸서 재실행 방지 |
| Graph API | `checkpointer` 연결로 노드별 자동 저장 |
| Functional API | `@entrypoint` + `@task`로 내구성 보장 |
| 장애 복구 | 같은 `thread_id`로 체크포인트에서 재개 |

### 다음 단계
→ **[13. API 선택 가이드와 Pregel](13_api_guide_and_pregel.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [Durable Execution](../docs/langgraph/06-durable-execution.md)